# 01. Thực nghiệm E0-E4 (P1-P4, cần GPU)

Mọi cấu hình dùng **cùng reader, cùng prompt hệ thống, cùng decoding (greedy), cùng ngân sách**: khối package (metadata + đoạn mã `C*`) tối đa
3000 token cho mọi method; bằng chứng thêm vào (`K*`, `G*`, `F*`) tối đa 1500 token cho E1-E4. Mọi cell **bỏ qua nếu `results.json` đã có**;
mọi câu trả lời của LLM được cache (`llm_cache.jsonl`), nên chạy lại hoặc bị ngắt giữa chừng không tốn lại GPU. E4 dùng lại đúng output của E3 (cùng prompt).

| Bước | Nội dung | RQ |
|---|---|---|
| 00 | Thiết lập, GPU, backend, cấu hình | - |
| 01 | Tái lập mal-LLM: zero-shot, RAG-YARA, RAG-malcode (300 ký tự đầu setup.py) | P1 |
| 02 | Validation: ablation retrieval, tune chính sách E4 | P2, P4 |
| 03 | D1 `malllm` test: E0, E1, E2 (+ BM25, rerank), E3, E3a, E4 (+ filter, cite) | RQ1-RQ4 |
| 04 | D2 `pyguard_obf` test (package đầy đủ) | RQ1-RQ4 |
| 05 | D3 `synthetic` (có gold): coverage bằng chứng, recall truy hồi, path recall | mục 12.2 |
| 06 | Chi phí, độ lặp lại | mục 12.5 |
| 07 | Checklist nghiệm thu (plan mục 15) | - |

**Khối lượng mặc định** (2 reader, 500 test + 200 val mỗi bộ): khoảng 19.000 lời gọi dài (~3.500 token vào, ~250 token ra) và 3.000 lời gọi ngắn
cho bước tái lập. Ước lượng (chưa đo, tính từ thông lượng điển hình của model 8B bf16):

| Backend / GPU | Thời gian notebook này |
|---|---|
| `hf` (transformers.generate, batch 8), A100 40GB | 8-10 giờ |
| `vllm`, A100 40GB (Colab) | 2-3 giờ |
| `vllm`, 1x H100 80GB | 1-1.5 giờ |
| `vllm`, 2x H100 80GB, mỗi reader một GPU (`scripts/run_cloud.sh`) | 40-60 phút |

`vllm` nhanh hơn vì batching liên tục và cache prefix: các prompt của cùng một package (cùng khối package, khác phần bằng chứng)
được gửi liền nhau nên phần chung chỉ tính một lần. Dòng `[llm] i/n ... eta` trong log cho thời gian thật sau vài phút.

| Method | Thêm gì vào prompt |
|---|---|
| `e0` LLM-only | không thêm gì |
| `e1` Simple RAG | top-5 tài liệu KB theo embedding |
| `e2` Hybrid RAG | dense + BM25, reciprocal-rank fusion (`e2_bm25`, `e2_rerank` là ablation) |
| `e3` KG-RAG | đường đi 1-3 bước trong đồ thị hành vi khớp pattern, kèm file:dòng và provenance từng cạnh |
| `e3a` đối chứng | **cùng các cạnh** viết thành câu rời, chọn bằng hybrid retrieval, không duyệt đồ thị |
| `e4` Verified KG-RAG | E3 + kiểm claim theo fact tĩnh, bỏ claim không có chứng cứ, abstain khi thiếu/mâu thuẫn (`e4_filter`, `e4_cite` là ablation) |

In [ ]:
# ==== 00a. Mount Drive + đường dẫn + helper ====
import os, sys, json, subprocess, time, shutil, glob, random
try:
    from google.colab import drive; drive.mount('/content/drive'); IN_COLAB = True
except ImportError:
    IN_COLAB = False                                        # chạy local để kiểm tra notebook (CPU)

DRIVE_BASE = '/content/drive/MyDrive/ThS/Research/colab' if IN_COLAB else os.environ.get('PYPI_RAG_BASE', os.path.abspath('../..'))  # SỬA NẾU KHÁC
CODE_DIR = f'{DRIVE_BASE}/pypi_malware_rag'                 # folder code này (upload nguyên folder lên Drive)
WORK     = os.environ.get('PYPI_RAG_WORK', f'{DRIVE_BASE}/pypi_rag_work')   # dữ liệu chuẩn hoá, KB, kết quả (giữ qua các phiên)
EXT      = '/content/external' if IN_COLAB else f'{WORK}/external'          # repo upstream + package nghi độc: đĩa máy ảo, KHÔNG để trên Drive
QUICK    = os.environ.get('PYPI_RAG_QUICK') == '1'          # 1 = kiểm tra nhanh: chỉ dữ liệu tổng hợp, reader dummy, không tải gì

os.environ['PYTHONDONTWRITEBYTECODE'] = '1'; os.environ['PYTHONIOENCODING'] = 'utf-8'; sys.dont_write_bytecode = True
for d in glob.glob(f'{CODE_DIR}/**/__pycache__', recursive=True): shutil.rmtree(d, ignore_errors=True)
for d in (WORK, f'{WORK}/data', f'{WORK}/kb', f'{WORK}/results', EXT): os.makedirs(d, exist_ok=True)
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
assert os.path.exists('pypi_rag/cli.py'), 'chưa upload folder pypi_malware_rag lên Drive (hoặc sai DRIVE_BASE)'
for m in [m for m in sys.modules if m.split('.')[0] == 'pypi_rag']: del sys.modules[m]

CLI = f'{sys.executable} -m pypi_rag.cli'
def sh(cmd, check=True):
    """Chạy lệnh, in output trực tiếp; lỗi -> exception (check=True)."""
    print('$', cmd[:500]); t0 = time.time()
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         encoding='utf-8', errors='replace', cwd=CODE_DIR)
    for line in p.stdout: print(line, end='')
    rc = p.wait(); print(f'exit={rc}  ({(time.time() - t0) / 60:.1f} phút)')
    if check and rc: raise RuntimeError(f'lệnh lỗi (exit {rc}): {cmd[:200]}')
    return rc
def gate(cond, msg): print(('PASS  ' if cond else 'FAIL  ') + msg)
def load(out): return json.load(open(f'{out}/results.json', encoding='utf-8'))
import numpy as np, pandas as pd
pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 200)
print('code =', CODE_DIR, '| work =', WORK, '| colab =', IN_COLAB, '| quick =', QUICK)

In [ ]:
# ==== 00b. GPU + HF token + backend ====
BACKEND = os.environ.get('PYPI_RAG_BACKEND', 'vllm')   # 'vllm' (nhanh nhất) | 'hf' (transformers, chậm hơn 4-8 lần, không cần cài thêm)
if IN_COLAB:
    sh(f'{sys.executable} -m pip install -q "sentence-transformers>=2.2" "transformers>=4.43" accelerate networkx pyyaml')
if BACKEND == 'vllm' and not QUICK:
    ok = lambda: subprocess.call([sys.executable, '-c', 'import vllm'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL) == 0
    if not ok():
        sh(f'{sys.executable} -m pip install -q vllm', check=False)   # lần đầu 3-5 phút; nếu sau đó kernel báo lỗi thư viện: Restart session và chạy lại
    if not ok():
        print('[vllm] không cài được -> dùng backend hf'); BACKEND = 'hf'
print('backend =', BACKEND)
if IN_COLAB:
    try:
        from google.colab import userdata
        if userdata.get('HF_TOKEN'): os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN'); print('HF_TOKEN set')
    except Exception: print('(không có secret HF_TOKEN: dùng mirror NousResearch cho Llama)')
try:
    import torch
    print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
    if not QUICK: assert torch.cuda.is_available(), 'cần GPU: Runtime > Change runtime type > A100'
except ImportError:
    print('(không có torch: chỉ chạy được QUICK mode)')

In [ ]:
# ==== 00c. Cấu hình + helper ====
assert os.path.exists(f'{WORK}/FROZEN.json'), 'chưa chạy notebook 00 (dữ liệu, KB, đóng băng)'
READERS = ['NousResearch/Meta-Llama-3.1-8B-Instruct',   # reader của paper (Llama-3.1-8B), dùng để tái lập
           'Qwen/Qwen2.5-Coder-7B-Instruct']             # reader chuyên code
EMB, RERANK = 'BAAI/bge-small-en-v1.5', 'BAAI/bge-reranker-base'
if os.environ.get('PYPI_RAG_READERS'):                   # chạy song song: mỗi tiến trình một reader, một GPU (scripts/run_cloud.sh)
    READERS = os.environ['PYPI_RAG_READERS'].split(',')
MAX_TEST, MAX_VAL, SUBSET_SEED, BATCH = 500, 200, 0, 8   # None = toàn bộ split; cùng seed -> cùng sample id cho mọi method; BATCH chỉ dùng cho hf
METHODS_MAIN = ['e0', 'e1', 'e2', 'e2_bm25', 'e2_rerank', 'e3', 'e3a_exact', 'e3a', 'e4', 'e4_filter', 'e4_cite']
ONLY = [x for x in os.environ.get('PYPI_RAG_DATASETS', '').split(',') if x]   # nhiều máy: mỗi máy một bộ dữ liệu
EXTRAS = os.environ.get('PYPI_RAG_EXTRAS', '1') == '1'                         # synthetic, cases, lặp lại: chỉ một máy chạy
MAIN_DATA = [d for d in ('malllm', 'pyguard_obf') if os.path.exists(f'{WORK}/data/{d}/samples.jsonl.gz') and (not ONLY or d in ONLY)]
if QUICK:
    READERS, EMB, MAX_TEST, MAX_VAL, BATCH = ['dummy'], 'hash', None, None, 4
    METHODS_MAIN = [m for m in METHODS_MAIN if m != 'e2_rerank']; MAIN_DATA = ['synthetic']
BUNDLE = f'{WORK}/kb/pyguard_bundle.json' if os.path.exists(f'{WORK}/kb/pyguard_bundle.json') else None
def tag(rd): return rd.split('/')[-1]
def llm_of(rd): return 'dummy' if rd == 'dummy' else f'{BACKEND}:{rd}'
def kb_of(data): return f'{WORK}/kb/{data}.jsonl'

def run_bench(out, data, split, methods, llm, kb=None, extra='', max_items=None, cache=None, force=False):
    """CLI run; bỏ qua nếu results.json đã có đủ method. Cache LLM nằm trong `out` (hoặc `cache`)."""
    if os.path.exists(f'{out}/results.json') and not force:
        R = load(out); missing = [m for m in methods if m not in R['summaries']]
        if not missing:
            print(f'[skip] {os.path.basename(out)} đã có results.json'); return R
        print(f'[rerun] {os.path.basename(out)} thiếu {missing}; method cũ lấy từ cache LLM, không tốn GPU')
    cmd = (f'{CLI} run --config configs/experiments.yaml configs/retrieval.yaml configs/graph.yaml '
           f'--data {WORK}/data/{data} --split {split} --methods {" ".join(methods)} --llm {llm} --embedder {EMB} '
           f'--batch-size {BATCH} --out {out} --llm-cache {cache or out + "/llm_cache.jsonl"} --quiet '
           + ('' if QUICK else f'--reranker {RERANK} ') + (f'--kb {kb} ' if kb else '')
           + (f'--max-items {max_items} --subset-seed {SUBSET_SEED} ' if max_items else '')
           + (f'--pyguard-bundle {BUNDLE} ' if BUNDLE else '') + extra)
    sh(cmd); return load(out)

COLS = ['strict/f1_malicious', 'strict/macro_f1', 'strict/fpr', 'strict/fnr', 'coverage', 'aurc', 'answered/f1_malicious',
        'evidence/unsupported_claim_rate', 'evidence/unsupported_over_original_claims', 'evidence/items_with_unsupported_all',
        'evidence/citation_accuracy', 'tokens_in_mean', 'tokens_out_mean', 'latency_p50_ms', 'parse_failure_rate']
def show(R, cols=COLS, methods=None):
    df = pd.DataFrame(R['summaries']).T
    if methods: df = df.loc[[m for m in methods if m in df.index]]
    display(df[[c for c in cols if c in df.columns]].astype(float).round(3))
def show_cmp(R):
    rows = []
    for k, c in R['comparisons'].items():
        b = c['f1_malicious_strict']
        rows.append(dict(pair=k, question=c.get('question', ''), n=c['n'], dF1=round(b['delta'], 3),
                         ci=f"[{b['ci_low']:+.3f}, {b['ci_high']:+.3f}]", p=round(b['p_value'], 4),
                         dFPR=round(c['fpr_strict']['delta'], 3), mcnemar_p=round(c['mcnemar']['p_value'], 4), sig=c['significant_0.05']))
    display(pd.DataFrame(rows))
def rq_gates(R, label):
    """Kiểm định giả thuyết: FAIL nghĩa là dữ liệu KHÔNG ủng hộ giả thuyết, không phải lỗi code."""
    C, S = R['comparisons'], R['summaries']
    for k, txt in [('e1_vs_e0', 'RQ1: E1 (RAG) > E0 (LLM thuần)'), ('e2_vs_e1', 'RQ2: E2 (hybrid) > E1 (dense)'),
                   ('e3_vs_e2', 'RQ3: E3 (KG-RAG) > E2 (hybrid)'),
                   ('e3_vs_e3a_exact', 'RQ3 công bằng (E3a): cấu trúc đường > đúng các cạnh đó dạng câu rời'),
                   ('e3_vs_e3a', 'RQ3 (E3b): duyệt đồ thị > fact chọn bằng truy hồi')]:
        b = C.get(k, {}).get('f1_malicious_strict')
        if b: gate(b['ci_low'] > 0, f"{label} | {txt}: dF1={b['delta']:+.3f} CI=[{b['ci_low']:+.3f}, {b['ci_high']:+.3f}]")
    if 'e4' in S and 'e3' in S:
        G = R.get('grounding_vs_e3') or {}
        if G.get('e4'):          # cùng mẫu số: claim gốc của E3 và mọi mẫu, kể cả mẫu E4 từ chối
            u3, u4 = G['e3']['unsupported_over_original_claims'], G['e4']['unsupported_over_original_claims']
            gate(u4 < u3, f"{label} | RQ4a: claim không chứng cứ / claim gốc: E3 {u3:.3f} -> E4 {u4:.3f} "
                          f"(claim giữ lại {G['e4']['claims_retained']:.2f}, coverage {G['e4']['coverage']:.2f})")
        b = C.get('e4_vs_e3', {}).get('f1_malicious_strict')
        if b: gate(b['ci_low'] > -0.03, f"{label} | RQ4b: E4 không làm F1 strict giảm quá 3 điểm (dF1={b['delta']:+.3f}, coverage {S['e4']['coverage']:.2f})")
print('readers =', READERS, '| data =', MAIN_DATA, '| MAX_TEST =', MAX_TEST)

## 01. Tái lập baseline mal-LLM (P1)
Cùng input như mal-LLM: tên package, danh sách file, **300 ký tự đầu của `setup.py`**, prompt chuyển gần nguyên văn từ `simulate_test.py` / `simulate_yara_rag.py`.
RAG-YARA lấy top-2 luật YARA, RAG-malcode lấy top-1 setup.py độc của train, đều bằng embedding.

Số tham chiếu (Llama-3.1-8B, lấy từ output notebook `Results/analyze_result` của repo mal-LLM; **đối chiếu lại với PDF paper trước khi trích**):

| Cấu hình | Acc | P | R | F1 |
|---|---|---|---|---|
| Zero-shot | 0.708 | 0.466 | 0.462 | 0.459 |
| RAG-YARA | 0.721 | 0.484 | 0.551 | 0.515 |
| RAG-malcode | 0.604 | 0.352 | 0.579 | 0.438 |

Khác biệt có chủ đích hoặc không tránh được: Llama chạy local bf16 thay vì HF Inference API; embedding `bge-small` thay vì OpenAI ada-002; YARA bản mẫu
(trừ khi bật `YARA_FULL` ở notebook 00); luật YARA không được gpt-4o-mini diễn giải. Kết luận tái lập: |dF1| <= 0.05 *xấp xỉ*, <= 0.15 *lệch vừa*, lớn hơn *không tái lập được*.

In [ ]:
# ==== 01. Reproduction ====
PAPER = {'e0_malllm': dict(acc=.708, p=.466, r=.462, f1=.459), 'e1_malllm_yara': dict(acc=.721, p=.484, r=.551, f1=.515),
         'e1_malllm_code': dict(acc=.604, p=.352, r=.579, f1=.438)}
rd = next((r for r in READERS if 'llama' in r.lower()), READERS[0] if QUICK else None)   # paper dùng Llama-3.1-8B
data = 'malllm' if 'malllm' in MAIN_DATA else MAIN_DATA[0]
def reproduce(rd, data):
    R01 = run_bench(f'{WORK}/results/r01_repro_{data}_{tag(rd)}', data, 'test', list(PAPER), llm_of(rd), kb=kb_of(data),
                    extra='--max-new-tokens 200')
    rows = []
    for m, p in PAPER.items():
        s = R01['summaries'][m]
        ours = dict(acc=s['strict/accuracy'], p=s['strict/precision_malicious'], r=s['strict/recall_malicious'], f1=s['strict/f1_malicious'])
        d = abs(ours['f1'] - p['f1'])
        verdict = 'xấp xỉ' if d <= 0.05 else ('lệch vừa' if d <= 0.15 else 'không tái lập được')
        rows.append({'method': m, **{f'paper_{k}': v for k, v in p.items()}, **{f'ours_{k}': round(v, 3) for k, v in ours.items()},
                     'coverage': round(s['coverage'], 3), 'parse_fail': round(s['parse_failure_rate'], 3), 'kết luận': verdict})
    display(pd.DataFrame(rows).set_index('method'))
    json.dump(rows, open(f'{WORK}/results/r01_repro_{data}_{tag(rd)}/reproduction.json', 'w'), indent=2)
if rd is None or not (QUICK or 'malllm' in MAIN_DATA):
    print('bỏ qua bước tái lập (cần reader Llama và bộ malllm; máy/tiến trình khác chạy bước này)')
else:
    reproduce(rd, data)

## 02. Validation: ablation retrieval và tune chính sách E4 (plan mục 9, mục 11)
Chỉ dùng split `val`. Chính sách abstain của E4 (số claim nguy hiểm tối thiểu phải được chứng cứ hỗ trợ, ngưỡng mâu thuẫn với đường đồ thị)
được chọn theo malicious-F1 **strict** (abstain tính là sai), nên không thể thắng bằng cách bỏ cuộc. Muốn đổi `final_k` hay embedding thì sửa
`configs/retrieval.yaml` **trước** bước 03 và chạy lại bước này.

In [ ]:
# ==== 02. Validation ====
POLICY = {}
VAL_METHODS = ['e1', 'e2', 'e2_bm25'] + ([] if QUICK else ['e2_rerank']) + ['e3']
for data in MAIN_DATA:
    for rd in READERS:
        out = f'{WORK}/results/r02_val_{data}_{tag(rd)}'
        R = run_bench(out, data, 'val', VAL_METHODS, llm_of(rd), kb=kb_of(data), max_items=MAX_VAL)
        print(f'\n##### VAL {data} | {tag(rd)}'); show(R, cols=['strict/f1_malicious', 'strict/fpr', 'coverage', 'tokens_in_mean'])
        pol = f'{out}/policy.json'
        if not os.path.exists(pol):
            sh(f'{CLI} tune-policy --results {out} --data {WORK}/data/{data} --split val --method e3 --out {pol}')
        POLICY[(data, rd)] = pol; print('chính sách E4:', json.load(open(pol)))

## 03. D1 `malllm` (setup.py, test khoá theo paper)
Hai bảng: test đầy đủ (so được với paper) và test bỏ các mẫu trùng nhóm với train (`_nodup`, lấy từ cache, không tốn GPU).

In [ ]:
# ==== 03. D1 malllm ====
R_MAIN = {}
if 'malllm' in MAIN_DATA or QUICK:
    data = 'malllm' if 'malllm' in MAIN_DATA else 'synthetic'
    for rd in READERS:
        out = f'{WORK}/results/r03_test_{data}_{tag(rd)}'
        R = run_bench(out, data, 'test', METHODS_MAIN, llm_of(rd), kb=kb_of(data), max_items=MAX_TEST,
                      extra=f'--policy {POLICY[(data, rd)]}')
        R_MAIN[(data, rd)] = R
        print(f'\n##### TEST {data} | {tag(rd)} | n = {R["n_samples"]}'); show(R); show_cmp(R); rq_gates(R, f'{data}/{tag(rd)}')
        if data == 'malllm':
            Rn = run_bench(out + '_nodup', data, 'test', METHODS_MAIN, llm_of(rd), kb=kb_of(data), max_items=MAX_TEST,
                           cache=f'{out}/llm_cache.jsonl', extra=f'--policy {POLICY[(data, rd)]} --drop-cross-dups')
            print(f'--- bỏ mẫu trùng nhóm với train: n = {Rn["n_samples"]}'); show(Rn, cols=['strict/f1_malicious', 'strict/fpr', 'coverage'])

## 04. D2 `pyguard_obf` (package đầy đủ, nhiều file)
Đây là bộ mà quan hệ giữa các hành vi (đọc dữ liệu ở file này, gửi đi ở file khác) có cơ hội xuất hiện, nên là phép thử chính cho RQ3.

In [ ]:
# ==== 04. D2 pyguard_obf ====
if 'pyguard_obf' in MAIN_DATA:
    data = 'pyguard_obf'
    for rd in READERS:
        out = f'{WORK}/results/r04_test_{data}_{tag(rd)}'
        R = run_bench(out, data, 'test', METHODS_MAIN, llm_of(rd), kb=kb_of(data), max_items=MAX_TEST,
                      extra=f'--policy {POLICY[(data, rd)]}')
        R_MAIN[(data, rd)] = R
        print(f'\n##### TEST {data} | {tag(rd)} | n = {R["n_samples"]}'); show(R); show_cmp(R); rq_gates(R, f'{data}/{tag(rd)}')
        display(pd.DataFrame(R['strata']['e3']).T)
else:
    print('(không có pyguard_obf: bật USE_PYGUARD_OBF ở notebook 00)')

## 05. D3 `synthetic`: đo có gold (plan mục 12.2)
Fixture tổng hợp có `gold_behaviors` và `gold_paths`, nên đo được: **evidence coverage** (hành vi gold xuất hiện trong bằng chứng model thấy),
**recall@k / MRR** của KB, **path recall** của E3. Không dùng bộ này cho kết luận về mã độc thật.

In [ ]:
# ==== 05. D3 synthetic (gold) ====
GOLD = ['gold/evidence_coverage', 'gold/kb_recall@1', 'gold/kb_recall@3', 'gold/kb_recall@5', 'gold/kb_mrr', 'gold/path_recall',
        'strict/f1_malicious', 'strict/fpr', 'evidence/unsupported_claim_rate']
for rd in (READERS if EXTRAS else []):
    out = f'{WORK}/results/r05_test_synthetic_{tag(rd)}'
    R = run_bench(out, 'synthetic', 'test', [m for m in METHODS_MAIN if m != 'e2_rerank'], llm_of(rd), kb=kb_of('synthetic'))
    print(f'\n##### synthetic | {tag(rd)}'); show(R, cols=GOLD)
    display(pd.DataFrame({m: {g: v['accuracy_strict'] for g, v in st.items()} for m, st in R['strata'].items()}).round(2))

## 05b. D4 sáu test case kiểm soát C01-C06
Cùng 7 method trên 6 case của tài liệu test case. Hai cách hỏi: `q1` (câu hỏi chuẩn "độc hay lành", giống benchmark) và `qcase`
(câu hỏi riêng của từng case, `--use-sample-question`). KB của case chỉ gồm luật dựng sẵn và 5 bản ghi của tài liệu
(PATTERN-EXFIL-001, MAL-001, BEN-001, BEN-002, RULE-001), cố định trước khi chạy. E4 dùng chính sách đã tune trên validation,
không chỉnh riêng cho case. C06 có nhãn `unknown`, kỳ vọng `uncertain`: chỉ tính ở abstention accuracy.
Đây là minh hoạ có kiểm soát (6 mẫu), không phải bằng chứng thống kê.

In [ ]:
# ==== 05b. D4 cases C01-C06 ====
CASE_METHODS = ['e0', 'e1', 'e2', 'e3', 'e3a_exact', 'e3a', 'e4']
CASE_COLS = ['case/expected_verdict_accuracy', 'case/abstention_accuracy', 'case/false_abstention_rate',
             'case/required_evidence_recall_visible', 'case/required_evidence_recall_cited', 'case/answer_path_accuracy',
             'evidence/unsupported_claim_rate']
CASES_OK = os.path.exists(f'{WORK}/data/cases/samples.jsonl.gz')
for rd in (READERS if (EXTRAS and CASES_OK) else []):
    pol = next((POLICY[k] for k in POLICY if k[1] == rd), None)    # tune trên validation trước khi chạy case
    for qv, qflag in (('q1', ''), ('qcase', '--use-sample-question')):
        out = f'{WORK}/results/r07_cases_{qv}_{tag(rd)}'
        R = run_bench(out, 'cases', 'test', CASE_METHODS, llm_of(rd), kb=kb_of('cases'),
                      extra=f'{qflag} --strata-key case --n-bootstrap 200' + (f' --policy {pol}' if pol else ''))
        print(f'\n##### cases {qv} | {tag(rd)}'); show(R, cols=CASE_COLS)
        T = pd.DataFrame(R.get('cases') or [])
        if len(T):
            display(T.pivot(index='case', columns='method', values='verdict'))
            display(T.pivot(index='case', columns='method', values='req_evidence_cited').round(2))
if not CASES_OK: print('(chưa có data/cases: chạy lại notebook 00)')

## 06. Chi phí và độ lặp lại (plan mục 12.5)
Token vào/ra, độ trễ p50/p95 (đo khi sinh, cache giữ nguyên số đo), token cho 100 package. E0 chạy hai lần với cache riêng để kiểm tra
decoding greedy cho cùng kết quả (khác biệt còn lại thường do batch/padding với bf16).

In [ ]:
# ==== 06. Cost + repeatability ====
rows = []
for (data, rd), R in R_MAIN.items():
    for m, s in R['summaries'].items():
        rows.append(dict(data=data, reader=tag(rd), method=m, tokens_in=s['tokens_in_mean'], tokens_out=s['tokens_out_mean'],
                         p50_ms=s['latency_p50_ms'], p95_ms=s['latency_p95_ms'], tokens_per_100_pkgs=s['tokens_per_100_pkgs'],
                         f1_strict=s['strict/f1_malicious']))
display(pd.DataFrame(rows).round(1))
data, rd = MAIN_DATA[0], READERS[-1]
reps = []
for k in ((1, 2) if EXTRAS else ()):
    out = f'{WORK}/results/r06_repeat{k}_{data}_{tag(rd)}'
    run_bench(out, data, 'test', ['e0'], llm_of(rd), max_items=None if QUICK else 100)
    reps.append({json.loads(l)['sample_id']: json.loads(l)['verdict'] for l in open(f'{out}/predictions_test_e0.jsonl', encoding='utf-8')})
if reps:
    agree = np.mean([reps[0][i] == reps[1].get(i) for i in reps[0]])
    gate(agree >= 0.98, f'E0 lặp lại 2 lần (cache riêng): {agree:.3f} verdict trùng nhau')
if IN_COLAB: sh('nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv', check=False)

## 07. Checklist nghiệm thu (plan mục 15)
Đọc thẳng file trên Drive, không phụ thuộc cell nào đã chạy trong phiên này. `[OK]` đạt, `[NO]` không đạt (phải nói rõ là giới hạn), `[--]` chưa có kết quả.

In [ ]:
# ==== 07. Acceptance checklist ====
def _j(p):
    try: return json.load(open(p, encoding='utf-8'))
    except (FileNotFoundError, json.JSONDecodeError): return None
runs = sorted(d for d in glob.glob(f'{WORK}/results/r0[34]_test_*') if _j(f'{d}/results.json') and not d.endswith('_nodup'))
Rs = {os.path.basename(d): _j(f'{d}/results.json') for d in runs}
aud, frz = _j(f'{WORK}/upstream_audit.json'), _j(f'{WORK}/FROZEN.json')
kbr = [_j(p) for p in glob.glob(f'{WORK}/kb/*.jsonl.report.json')]
graphs = [_j(p) for p in glob.glob(f'{WORK}/graphs/*/graphs_test.summary.json')]
def same_ids(d):
    R = _j(f'{d}/results.json'); sp = R['config']['split']
    ids = [tuple(json.loads(l)['sample_id'] for l in open(f'{d}/predictions_{sp}_{m}.jsonl', encoding='utf-8')) for m in R['summaries']]
    return len(set(ids)) == 1
items = [
    ('Hai repo upstream được audit, pin SHA, ghi license', bool(aud) and all(v.get('sha') for v in aud.values() if isinstance(v, dict)) if aud else None),
    ('Manifest có hash, nguồn, nguồn nhãn, split đóng băng', bool(frz) if frz is not None else None),
    ('Không rò rỉ val/test vào KB', all(r['leakage_after']['clean'] for r in kbr) if kbr else None),
    ('E0/E1 tái lập hoặc ghi rõ khoảng cách', bool(glob.glob(f'{WORK}/results/r01_*/reproduction.json')) or None),
    ('E2 có ablation lexical / vector / rerank', all({'e1', 'e2_bm25', 'e2'} <= set(R['summaries']) for R in Rs.values()) if Rs else None),
    ('Cạnh và đường E3 trỏ về đúng vị trí mã', all(g['missing_provenance_edges'] == 0 for g in graphs) if graphs else None),
    ('So sánh E3 với E3a (cùng cạnh, câu rời) và E3b (fact chọn bằng truy hồi)',
     all({'e3_vs_e3a_exact', 'e3_vs_e3a'} <= set(R['comparisons']) for R in Rs.values()) if Rs else None),
    ('Claim của E4 được kiểm bằng tập gán nhãn thủ công', bool(glob.glob(f'{WORK}/review/claims_review_*.csv')) or None),
    ('E0-E4 cùng sample id, cùng reader, cùng ngân sách', all(same_ids(d) for d in runs) if runs else None),
    ('Đã báo F1/FPR, hallucination, coverage, chi phí', all('evidence/unsupported_claim_rate' in s and 'tokens_in_mean' in s
                                                            for R in Rs.values() for s in R['summaries'].values()) if Rs else None),
    ('Ít nhất một ca lỗi cho mỗi method (notebook 02)', bool(glob.glob(f'{WORK}/results/*/failure_cases.json')) or None),
    ('Không cài / chạy package nghi độc (đọc tĩnh, trong bộ nhớ)', True),
]
_tag = {True: '[OK] ', False: '[NO] ', None: '[--] '}
for name, ok in items: print(_tag[None if ok is None else bool(ok)] + name)